# 03 — Raster Workflows: Labels in GeoTIFF Metadata

**Repo:** local_contexts_geospatial  
**Author:** Lilly Jones, PhD — Daear Consulting, LLC  

---

## What This Notebook Covers

Raster data — satellite imagery, elevation models, NDVI composites —
is the dominant format for environmental data about Indigenous lands.
GeoTIFF is the standard raster format, and it supports arbitrary
metadata tags that can carry TK/BC labels.

This notebook shows:
1. How to embed TK/BC labels in GeoTIFF metadata tags using `rasterio`
2. How labels survive GeoTIFF read/write cycles
3. How to propagate labels from a source raster to derived products
4. How to write a sidecar `.json` metadata file alongside the raster
5. The xarray/rioxarray approach for labeled multi-variable rasters

## Data

All rasters in this notebook are synthetic — created programmatically
to demonstrate the labeling patterns without requiring data downloads.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS

from localcontexts.labels import TKLabel, TKMetadata, extract_tk_fields, has_any_label
from localcontexts.propagation import propagate_labels, add_provenance_step
from localcontexts.validation import validate_usage, validate_export_ready

warnings.filterwarnings("ignore")
%matplotlib inline

SYNTH_DIR = REPO_ROOT / "data" / "synthetic"
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print(f"rasterio version: {rasterio.__version__}")
print("Setup complete.")

---
## Step 1: Create a Synthetic Labeled Raster

We create a synthetic NDVI raster covering Pine Ridge, then
embed a TK label in its metadata tags.

In [ ]:
# Define the TK label for this raster
tk = TKMetadata(
    label     = TKLabel.NON_COMMERCIAL,
    community = "Oglala Lakota Nation",
    authority = "Tribal Data Governance Office",
    usage     = "Non-commercial environmental research only",
    contact   = "data@oglalalakota.org",
)

# Pine Ridge bounding box (approximate)
WEST, SOUTH, EAST, NORTH = -103.5, 42.5, -101.5, 43.8

# Synthetic NDVI data
np.random.seed(7)
HEIGHT, WIDTH = 64, 64
ndvi_data = np.random.uniform(0.2, 0.65, (HEIGHT, WIDTH)).astype(np.float32)
# Add a spatial gradient — greener in the east (more moisture)
lon_gradient = np.linspace(0, 0.15, WIDTH)
ndvi_data += lon_gradient[np.newaxis, :]
ndvi_data = ndvi_data.clip(0, 1)

transform = from_bounds(WEST, SOUTH, EAST, NORTH, WIDTH, HEIGHT)

# Build the full metadata dict — TK label fields embedded alongside
# standard rasterio metadata
raster_meta = {
    "driver":   "GTiff",
    "dtype":    "float32",
    "width":    WIDTH,
    "height":   HEIGHT,
    "count":    1,
    "crs":      CRS.from_epsg(4326),
    "transform": transform,
    "nodata":   -9999.0,
}

# Attach TK label — returns a new dict with tk: fields added
labeled_meta = tk.attach(raster_meta)

print("Raster metadata with TK label:")
for k, v in labeled_meta.items():
    print(f"  {k}: {v}")

In [ ]:
# Write the labeled GeoTIFF
# rasterio tags are the mechanism for embedding custom metadata in GeoTIFF

tif_path = SYNTH_DIR / "pine_ridge_synthetic_ndvi.tif"

# Separate rasterio-native kwargs from TK label fields
rasterio_kwargs = {k: v for k, v in labeled_meta.items()
                   if not k.startswith("tk:")}
tk_tags = extract_tk_fields(labeled_meta)

with rasterio.open(tif_path, "w", **rasterio_kwargs) as dst:
    dst.write(ndvi_data, 1)
    dst.update_tags(**tk_tags)            # embed TK fields as GeoTIFF tags
    dst.update_tags(                      # standard metadata tags
        DESCRIPTION = "Synthetic NDVI — Pine Ridge, Oglala Lakota Nation",
        SOURCE      = "Synthetic data — demonstration only",
        CREATED     = "2024",
        IEEE_2890   = "https://standards.ieee.org/ieee/2890/10318/",
    )

print(f"GeoTIFF written: {tif_path}")
print(f"File size: {tif_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Verify labels survived the write/read cycle
with rasterio.open(tif_path) as src:
    tags = src.tags()
    data = src.read(1)

print("Tags read from GeoTIFF:")
for k, v in tags.items():
    print(f"  {k}: {v}")

print(f"\nData shape: {data.shape}")
print(f"TK label present: {'tk:label' in tags}")
print(f"TK label value: {tags.get('tk:label')}")

---
## Step 2: Derive a New Raster and Propagate the Label

In [ ]:
# Compute a drought stress proxy: invert NDVI so high = stressed
# The derived raster inherits the TK label from the source

def compute_stress_and_propagate(source_tif: Path, output_tif: Path):
    """Derive stress raster from NDVI and propagate TK label."""

    with rasterio.open(source_tif) as src:
        source_tags = src.tags()
        source_data = src.read(1)
        profile     = src.profile.copy()

    # Derive: stress = 1 - NDVI (low vegetation = high stress)
    stress_data = (1 - source_data).clip(0, 1).astype(np.float32)

    # Build output metadata dict
    output_tags = {
        "DESCRIPTION": "Drought stress proxy — derived from NDVI",
        "PROCESS":     "stress = 1 - ndvi",
        "SOURCE":      str(source_tif.name),
    }

    # Propagate TK label from source to derived product
    output_tags = propagate_labels(source_tags, output_tags)

    # Add provenance step
    output_tags = add_provenance_step(
        output_tags,
        process  = "NDVI inversion to drought stress proxy",
        source   = source_tif.name,
        workflow = "notebook_03_raster_workflows.ipynb",
    )

    # Write derived raster with propagated label
    tk_fields = extract_tk_fields(output_tags)
    other_tags = {k: str(v) for k, v in output_tags.items()
                  if not k.startswith("tk:") and k != "prov:chain"}

    with rasterio.open(output_tif, "w", **profile) as dst:
        dst.write(stress_data, 1)
        dst.update_tags(**tk_fields)
        dst.update_tags(**other_tags)

    return output_tags


stress_path = SYNTH_DIR / "pine_ridge_synthetic_stress.tif"
stress_tags = compute_stress_and_propagate(tif_path, stress_path)

print(f"Stress raster written: {stress_path}")
print(f"\nTK label propagated to derived product:")
print(f"  tk:label     = {stress_tags.get('tk:label')}")
print(f"  tk:community = {stress_tags.get('tk:community')}")
print(f"  prov:process = {stress_tags.get('prov:process')}")

---
## Step 3: Sidecar JSON Metadata File

In [ ]:
# Write a human- and machine-readable sidecar JSON alongside the raster
# Convention: same stem as the raster, .json extension

def write_sidecar(tif_path: Path) -> Path:
    """Write TK label and full metadata as a sidecar JSON file."""
    with rasterio.open(tif_path) as src:
        tags    = src.tags()
        profile = src.profile
        bounds  = src.bounds

    sidecar = {
        "file":           tif_path.name,
        "spatial_extent": {
            "west":  bounds.left,
            "south": bounds.bottom,
            "east":  bounds.right,
            "north": bounds.top,
        },
        "crs":            str(profile.get("crs")),
        "dimensions":     {"width": profile["width"], "height": profile["height"]},
        "metadata":       tags,
        "ieee_2890":      "https://standards.ieee.org/ieee/2890/10318/",
    }

    sidecar_path = tif_path.with_suffix(".json")
    sidecar_path.write_text(json.dumps(sidecar, indent=2))
    return sidecar_path


for path in [tif_path, stress_path]:
    sc = write_sidecar(path)
    print(f"Sidecar written: {sc.name}")

# Show the sidecar content
print("\nSidecar JSON content (NDVI raster):")
print(tif_path.with_suffix(".json").read_text()[:800], "...")

---
## Step 4: Validate Before Export

In [ ]:
# Run pre-export validation on the stress raster

print("Validation: research use")
report = validate_export_ready(
    stress_tags,
    intended_use = "research",
    destination  = "GitHub repository",
)
for check in report["checks"]:
    status = "✓" if check["passed"] else "✗"
    print(f"  {status} {check['check']}: {check['note']}")
print(f"  All passed: {report['all_passed']}")

print()
print("Validation: commercial use (should fail)")
try:
    validate_usage(stress_tags, intended_use="commercial", raise_on_violation=True)
except Exception as e:
    print(f"  ✗ Correctly blocked: {e}")

---
## Step 5: Visualization

In [ ]:
with rasterio.open(tif_path) as src:
    ndvi_arr = src.read(1)
    ndvi_tags = src.tags()
    ndvi_bounds = src.bounds

with rasterio.open(stress_path) as src:
    stress_arr = src.read(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
extent = [ndvi_bounds.left, ndvi_bounds.right, ndvi_bounds.bottom, ndvi_bounds.top]

im0 = axes[0].imshow(ndvi_arr, cmap="YlGn", vmin=0.1, vmax=0.85,
                      origin="upper", extent=extent)
plt.colorbar(im0, ax=axes[0], label="NDVI", shrink=0.8)
axes[0].set_title(
    f"Source NDVI\nTK: {ndvi_tags.get('tk:label', 'none')}",
    fontsize=10, fontweight="bold",
)

im1 = axes[1].imshow(stress_arr, cmap="YlOrRd", vmin=0, vmax=0.85,
                      origin="upper", extent=extent)
plt.colorbar(im1, ax=axes[1], label="Stress (1 - NDVI)", shrink=0.8)
axes[1].set_title(
    f"Derived Stress Proxy\nTK label propagated: {stress_tags.get('tk:label', 'none')}",
    fontsize=10, fontweight="bold",
)

for ax in axes:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.suptitle(
    "TK Label Travels from Source to Derived Raster\n"
    "Pine Ridge, Oglala Lakota Nation — Synthetic demonstration data",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
plt.show()

---
## Summary

| Step | rasterio method | What it does |
|---|---|---|
| Write label | `dst.update_tags(**tk_fields)` | Embeds TK fields as GeoTIFF tags |
| Read label | `src.tags()` | Returns all tags as a dict |
| Propagate | `propagate_labels(source_tags, output_tags)` | Copies tk: fields to derived product |
| Sidecar | `path.with_suffix(".json").write_text(...)` | Human-readable companion file |
| Validate | `validate_export_ready(tags, intended_use=...)` | Pre-export governance check |

---
## Next Notebook

**04 — Propagation and Lineage:** The critical notebook — what happens
when labels get dropped vs. correctly propagated through a workflow.